# NB07 — 정준 기술 통계 분석 (Canonical Descriptive EDA) 및 정확 분해

```
NOTEBOOK_ID       = 07_eda_and_decomposition
EXECUTION_BASE    = origin/audit/g5-analysis-readiness-20260818 @ 9d99e13026b89dcf7d8846d0c105a811f64274bc
BRANCH            = research/nb07-canonical-eda-20260818
G5_STATUS         = G5_ANALYSIS_READINESS_PASS_WITH_NOTES  (RD-G5-ENTRY-GOVERNANCE-CLOSEOUT-01 → G5 adjudication → G5 independent audit)
```

**성격.** 이 notebook은 **기술 통계(descriptive)** 전용이다. 어떤 모형도 적합하지 않고,
어떤 계수도 산출하지 않으며, RQ1을 제외한 어떤 RQ도 검정하지 않는다. RQ1은 NB08에서 이미
닫힌 결과를 **주석(annotate)** 하는 것으로 한정한다. RQ3/RQ4/RQ5/RQ6의 *조건부 설명력*,
*증분 설명력*, *메커니즘 기여*는 전부 NB09(M1 vs M0, M2 vs M1, M3 vs M2)의 소관이며 이
notebook은 그 입력이 되는 분포 구조만 기술한다.

**Authority 인계.**

```
KOEN-TP-RS-001                                  (SSOT parent authority)
RD-FAST-G5-01                                    (realized-model contract, M0-M3 ladder)
RD-SSOT-CANONICAL-RETURN-01                      (canonical return, NB07 role §14)
RD-G5-ENTRY-GOVERNANCE-CLOSEOUT-01               (governance lineage closeout)
ssot_g5/02_G5_DIAGNOSTIC_PROTOCOL_v001.md        (frozen §3/§4/§8 column definitions)
ssot/2026-08-18_1325_..._G5_ANALYSIS_READINESS_ADJUDICATION.md   (Claude-B G5 adjudication)
ssot/2026-08-18_1344_..._G5_INDEPENDENT_AUDIT.md                 (Claude-A G5 independent audit)
```

**과거 문서와의 관계.** V1(`results/preliminary-eda-v1-20260817`)과 V2
(`results/pre-g5-eda-v2-20260817`)는 역사적·비canonical 참고 자료다. NB08-RQ1-Vxx와
NB06_D05_Vxx는 결과-소통용 figure이지 canonical F01–F09가 아니다. 이 notebook의 모든 수치는
D-02~D-05 물리 artifact에서 **새로 계산**한다 — 과거 문서의 숫자를 복사하지 않는다. 과거
수치와 일치하면 그것은 교차검증(corroboration)이고, 불일치하면 그 불일치를 그대로 보고한다
(억지로 맞추지 않는다).

**레거시 notebook 처리.** `notebooks/exploratory/EDA_representation_kiwi_o200k_casebook.ipynb`는
`LEGACY_UNTRACKED_REFERENCE_ONLY`다. 편집·이름변경·수치 복사·해석 복사를 하지 않았으며, 이
notebook의 어떤 셀도 그 결과를 인용하지 않는다.


In [1]:

# ── 실행 환경 설정 ──────────────────────────────────────────────────────
from __future__ import annotations
import sys, os, json, time, hashlib, math
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import duckdb
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

from tokenization_premium.visualization import find_korean_font
from tokenization_premium.telemetry import RuntimeTelemetry

FONT_INFO = find_korean_font()
FP = font_manager.FontProperties(fname=FONT_INFO["path"])
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["svg.fonttype"] = "none"
print("KOREAN_PLOT_FONT=", FONT_INFO["family"], FONT_INFO["path"])

BASE = PROJECT_ROOT / "data" / "registry"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "nb07"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
MANIFEST_DIR = PROJECT_ROOT / "outputs" / "manifests"
for d in (FIG_DIR, REPORT_DIR, MANIFEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA memory_limit='6GB'")
(PROJECT_ROOT / ".runtime" / "nb07" / "duckdb-spill").mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory='{PROJECT_ROOT}/.runtime/nb07/duckdb-spill'")
con.execute("PRAGMA threads=8")

def save_fig(fig, name: str) -> dict:
    png = FIG_DIR / f"{name}.png"
    svg = FIG_DIR / f"{name}.svg"
    fig.savefig(png, dpi=150, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    def sha(p):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()
    return {"name": name, "png": str(png.relative_to(PROJECT_ROOT)),
            "svg": str(svg.relative_to(PROJECT_ROOT)),
            "png_sha256": sha(png), "svg_sha256": sha(svg)}

FIGURE_MANIFEST: dict[str, dict] = {}
SUMMARY: dict[str, object] = {}
ANOMALIES: list[dict] = []


KOREAN_PLOT_FONT= NanumGothic /usr/share/fonts/truetype/nanum/NanumGothic.ttf


## 1. Artifact identity 및 cohort freeze — fail closed

### SSOT / RQ 대응

Research Question:
NB07은 특정 RQ에 대한 검정이 아니라 D-02~D-05 canonical artifact 위에서 이후 모든 절의
분모가 되는 **artifact identity와 cohort 동일성**을 확인하는 절이다.

SSOT section:
§12.2 (D-02) · §12.3 (D-03) · §12.4 (D-04) · §12.5 (D-05) · §31 (G5 cohort freeze 요건)

Canonical input:
`data/registry/{REP_FEATURES_v002, MORPH_FEATURES_KIWI_v001, TOKEN_O200K_BASE_v001, CHUNK_O200K_BASE_v001}.parquet`,
`data/registry/PAIR_REGISTRY_v002.parquet` (D-01, source/domain/direction 메타데이터)

Physical variables:
SHA-256 identity, `pair_id` 집합, pair-set md5

Research purpose:
이후 모든 기술 통계가 동일한 3,835,988 pair 위에서 계산됨을 실행마다 재확인한다.

Allowed claim:
"이 notebook의 모든 수치는 해시가 검증된 D-02~D-05에서 새로 계산되었다."

Prohibited claim:
D-02~D-05의 재생성, 재추정, cohort 변경 — 이 notebook은 읽기 전용이다.


In [2]:

# fail-closed artifact identity — 이 dict과 다르면 즉시 AssertionError
EXPECTED_SHA256 = {
    "PAIR_REGISTRY_v002.parquet": "95f523d11b0e8fcfd761dee949f082e9b4590b919801441fbcfa3426010bec52",
    "REP_FEATURES_v002.parquet": "dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309",
    "MORPH_FEATURES_KIWI_v001.parquet": "0fe5bd74e3993a7141c5c33ea78e71b2c66e3ecd296544bde2615acb43e50f7d",
    "TOKEN_O200K_BASE_v001.parquet": "1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7",
    "CHUNK_O200K_BASE_v001.parquet": "bfa98bd6cf7ee8b7254c469aed3e259ce43cc8f0529153347ca4c2c3fc1944ab",
}
EXPECTED_N = 3_835_988
EXPECTED_PAIR_SET_MD5 = "d9660d654ee449e4d0c23a0070225274"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

identity_report = {}
for fname, expected in EXPECTED_SHA256.items():
    actual = sha256_file(BASE / fname)
    identity_report[fname] = {"expected": expected, "actual": actual, "match": actual == expected}
    assert actual == expected, f"CANONICAL_ARTIFACT_IDENTITY_MISMATCH: {fname}"
print("ARTIFACT_IDENTITY = 5/5 MATCH")

con.execute(f"""
CREATE OR REPLACE VIEW d01 AS SELECT * FROM read_parquet('{BASE}/PAIR_REGISTRY_v002.parquet');
CREATE OR REPLACE VIEW d02 AS SELECT * FROM read_parquet('{BASE}/REP_FEATURES_v002.parquet');
CREATE OR REPLACE VIEW d03 AS SELECT * FROM read_parquet('{BASE}/MORPH_FEATURES_KIWI_v001.parquet');
CREATE OR REPLACE VIEW d04 AS SELECT * FROM read_parquet('{BASE}/TOKEN_O200K_BASE_v001.parquet');
CREATE OR REPLACE VIEW d05 AS SELECT * FROM read_parquet('{BASE}/CHUNK_O200K_BASE_v001.parquet');
""")

n_rows = con.execute("SELECT COUNT(*) FROM d04").fetchone()[0]
n_distinct = con.execute("SELECT COUNT(DISTINCT pair_id) FROM d04").fetchone()[0]
pair_set_md5 = con.execute("SELECT md5(string_agg(pair_id, '' ORDER BY pair_id)) FROM d04").fetchone()[0]
assert n_rows == EXPECTED_N == n_distinct, f"COHORT_N_MISMATCH: {n_rows}"
assert pair_set_md5 == EXPECTED_PAIR_SET_MD5, f"PAIR_SET_MISMATCH: {pair_set_md5}"

for name in ("d01", "d02", "d03", "d05"):
    joined = con.execute(f"SELECT COUNT(*) FROM d04 t JOIN {name} s USING(pair_id)").fetchone()[0]
    assert joined == n_rows, f"JOIN_N_LOSS: d04 x {name} -> {joined}"

print(f"COHORT_N = {n_rows:,}  PAIR_SET_MD5 = {pair_set_md5}  ALL JOINS PRESERVE N")
SUMMARY["artifact_identity"] = identity_report
SUMMARY["cohort_n"] = n_rows
SUMMARY["pair_set_md5"] = pair_set_md5


ARTIFACT_IDENTITY = 5/5 MATCH


COHORT_N = 3,835,988  PAIR_SET_MD5 = d9660d654ee449e4d0c23a0070225274  ALL JOINS PRESERVE N


### 한국어 수치 해석

다섯 개 canonical artifact(D-01/D-02/D-03/D-04/D-05)의 SHA-256이 모두 기대값과 일치했고,
D-04(TOKEN_O200K_BASE_v001, N=3,835,988)를 기준으로 D-01/D-02/D-03/D-05를 join했을 때 행 손실이
없었다. pair-set md5(`d9660d654ee449e4d0c23a0070225274`)도 기대값과 일치한다.

**분포 해석**: 해당 없음 (identity check 절).

**연구적 의미**: 이후 모든 절의 분모가 fail-closed로 확정되었다. G5 adjudication/independent
audit이 보고한 동일 해시·동일 N·동일 pair-set을 이 notebook에서도 독립적으로 재확인했다.

**해석 한계**: 이 확인은 artifact의 *신원(identity)*만 검증하며, 각 컬럼의 의미론적 정확성은
G2~G4 forensic adjudication(`441d5802…`)의 소관이다. NB07은 그 adjudication을 재실행하지 않는다.


## 2. Cohort 구성 (Section A)

### SSOT / RQ 대응

Research Question:
없음 (기술 통계) — 이후 RQ3/RQ6 이질성 해석의 지지 구조를 제공.

SSOT section:
§20.2 (source/domain 식별성), §32 T-03/T-04 (contingency 요건)

Canonical input:
D-01 `source_id`, `domain`, `translation_direction` × D-04 `pair_id` spine

Physical variables:
`source_id`, `domain`, `translation_direction`, `length_stratum`

Research purpose:
관측된 source×domain×direction 지지 구조를 기술한다 — G5 ID-03/ID-04의 근거 자료.

Allowed claim:
"이 cohort에서 관측된 source-domain-direction 조합 빈도는 다음과 같다."

Prohibited claim:
순수 source 효과, 순수 domain 효과, 원인적 해석 — source와 domain은 이 cohort에서
분리 식별되지 않는다 (G5 ID-03).


In [3]:

cohort_composition = con.execute('''
    SELECT substr(source_id,1,3) AS source_short, domain, translation_direction, COUNT(*) AS n
    FROM d01 JOIN d04 USING(pair_id)
    GROUP BY substr(source_id,1,3), domain, translation_direction
    ORDER BY 1,2,3
''').fetchdf()
source_domain = con.execute('''
    SELECT substr(source_id,1,3) AS source_short, domain, COUNT(*) AS n
    FROM d01 JOIN d04 USING(pair_id)
    GROUP BY substr(source_id,1,3), domain ORDER BY 1,2
''').fetchdf()
direction_counts = con.execute('''
    SELECT translation_direction, COUNT(*) AS n
    FROM d01 JOIN d04 USING(pair_id) GROUP BY translation_direction ORDER BY n DESC
''').fetchdf()
print(source_domain.to_string(index=False))
print()
print(direction_counts.to_string(index=False))
SUMMARY["cohort_composition"] = {
    "source_domain_cells": source_domain.to_dict("records"),
    "direction_counts": direction_counts.to_dict("records"),
}


source_short     domain       n
         025   dialogue  516162
         025    general  804291
         025      other 1165510
         026      other  990120
         026 technology  359905

translation_direction       n
             KO_TO_EN 2512152
             EN_TO_KO 1273289
              UNKNOWN   50547


### 한국어 수치 해석

관측된 source×domain 조합은 5개뿐이다(- 025×dialogue: 516,162\n- 025×general: 804,291\n- 025×other: 1,165,510\n- 026×other: 990,120\n- 026×technology: 359,905). 15개 이론적 조합(2 source × 4 domain은
아니고, 실제로는 2 source × 여러 domain) 중 `other` domain만 두 source에 걸쳐 있고
`dialogue`/`general`은 025 전용, `technology`는 026 전용이다.

번역 방향(`translation_direction`) 분포:
- KO_TO_EN: 2,512,152 (65.49%)\n- EN_TO_KO: 1,273,289 (33.19%)\n- UNKNOWN: 50,547 (1.32%)

**분포 해석**: source-domain 지지 구조가 불균형하다 — 이는 G5 ID-03이 이미 형식적으로
판정한 "source와 domain의 독립 주효과는 이 cohort에서 분리 식별되지 않는다"는 사실의
기술적 확인이다.

**연구적 의미**: 이후 F03(source×domain descriptive TP)과 identifiability 절(ID-03/ID-04)의
직접적 입력이다.

**해석 한계**: 이 표는 지지 구조(support structure)만 보여준다. `source_domain_cell`
계수를 "순수 source 효과"나 "순수 domain 효과"로 해석하는 것은 금지된다(G5 ID-03).


## 3. KO / EN 토큰 수 (Section B, F01 dependency)

### SSOT / RQ 대응

Research Question:
RQ1의 원자료 — `T_KO,i`, `T_EN,i` 자체의 분포 기술.

SSOT section:
§12.4 (D-04 physical schema), §8 (TP 정의)

Canonical input:
D-04 `ko_token_count`, `en_token_count`

Physical variables:
`ko_token_count`, `en_token_count` (both, `o200k_base` Track A)

Research purpose:
TP를 구성하는 두 원시 수량 자체의 분포와 관계를 F01로 시각화한다.

Allowed claim:
관측된 토큰 수 분포와 KO/EN 간 관계의 기술.

Prohibited claim:
토큰 수 차이의 원인에 대한 주장 (그것은 D-05/NB09의 소관).


In [4]:

token_counts = con.execute('''
    SELECT median(ko_token_count) AS ko_median, avg(ko_token_count) AS ko_mean,
           min(ko_token_count) AS ko_min, max(ko_token_count) AS ko_max,
           median(en_token_count) AS en_median, avg(en_token_count) AS en_mean,
           min(en_token_count) AS en_min, max(en_token_count) AS en_max
    FROM d04
''').fetchdf()
print(token_counts.to_string(index=False))

ko_en_df = con.execute('''
    SELECT ko_token_count, en_token_count FROM d04
''').fetchdf()
H01, xe01, ye01 = np.histogram2d(ko_en_df["ko_token_count"], ko_en_df["en_token_count"],
                                  bins=100, range=[[0, 150], [0, 150]])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(xe01, ye01, np.log1p(H01.T), cmap="viridis", shading="auto")
ax.plot([0, 150], [0, 150], color="white", ls="--", lw=1, alpha=0.7)
ax.set_xlabel("한국어 토큰 수 (ko_token_count)", fontproperties=FP)
ax.set_ylabel("영어 토큰 수 (en_token_count)", fontproperties=FP)
ax.set_title("F01 — KO/EN 토큰 수 관계 (log(1+count) 밀도)", fontproperties=FP)
cb = fig.colorbar(im, ax=ax); cb.set_label("log(1+빈도)", fontproperties=FP)
FIGURE_MANIFEST["F01"] = save_fig(fig, "F01_token_count_relationship")
SUMMARY["token_counts"] = token_counts.to_dict("records")[0]


 ko_median   ko_mean  ko_min  ko_max  en_median   en_mean  en_min  en_max
      21.0 27.122633       1     401       16.0 20.170591       1     453


### 한국어 수치 해석

`ko_token_count`의 중앙값은 21, 평균 27.12이며,
`en_token_count`의 중앙값은 16, 평균 20.17이다. KO 평균이 EN
평균보다 약 1.345배 크다. 범위는 KO `[1, 401]`,
EN `[1, 453]`이다.

**분포 해석 (F01)**: 대부분의 질량이 대각선(KO=EN) 위쪽, 즉 KO > EN 영역에 놓여 있다 — 동일
의미를 나타내는 pair에서 한국어 측 토큰 수가 체계적으로 더 많다.

**연구적 의미**: TP = T_KO/T_EN의 raw 구성 요소를 직접 눈으로 확인시켜, RQ1의 결과(중앙값
TP=4/3)가 어떤 분포 위에서 나왔는지 보여준다.

**해석 한계**: 이 절은 두 변수의 raw 관계만 보이며, 어떤 언어학적 원인도 주장하지 않는다.


## 4. TP / logTP / ΔT 분포 및 RQ1 결과 주석 (Section C, F02)

### SSOT / RQ 대응

Research Question:
RQ1 — `Median(log_token_premium) > 0`인가. **이미 NB08에서 닫혔다
(`RQ1_PRIMARY_INFERENCE_PASS`, `NB08_RQ1_CLOSED`). 이 절은 그 결과를 재검정하지 않고
주석(annotate)한다.**

SSOT section:
§8 (TP/logTP 정의), §17.3 (대표본 해석 원칙)

Canonical input:
D-04 `token_premium`, `log_token_premium`, `token_difference`

Physical variables:
`token_premium`, `log_token_premium`, `token_difference`

Research purpose:
전집단 분포를 F02(히스토그램+ECDF)로 시각화하고, NB08이 보고한 정확 중앙값과
bit-identical함을 재확인한다.

Allowed claim:
분포 기술 + "이 notebook에서 독립적으로 재계산한 median(logTP)이 NB08 closeout의 값과
bit-identical하다."

Prohibited claim:
새로운 유의성 검정, 새로운 CI, RQ1 재검정 — 그것은 이미 닫힌 NB08의 소관이다.


In [5]:

tp_stats = con.execute('''
    SELECT median(token_premium) AS tp_median, median(log_token_premium) AS logtp_median,
           avg(log_token_premium) AS logtp_mean, stddev(log_token_premium) AS logtp_sd,
           median(token_difference) AS dt_median, avg(token_difference) AS dt_mean,
           sum(CASE WHEN token_premium > 1 THEN 1 ELSE 0 END) AS n_gt1,
           sum(CASE WHEN token_premium < 1 THEN 1 ELSE 0 END) AS n_lt1,
           sum(CASE WHEN token_premium = 1 THEN 1 ELSE 0 END) AS n_eq1,
           count(*) AS n
    FROM d04
''').fetchdf().iloc[0]
print(tp_stats)

RQ1_FROZEN_MEDIAN_LOGTP = 0.28768207245178085  # ssot_nb01/04_NB08_RQ1_RESULTS_v001.json (frozen)
match = abs(tp_stats["logtp_median"] - RQ1_FROZEN_MEDIAN_LOGTP) < 1e-12
print("RQ1_ANNOTATION_MATCH_BIT_IDENTICAL =", match, "| ln(4/3) =", math.log(4/3))
assert match, "RQ1 frozen median does not match fresh full-population recomputation"

logtp = con.execute("SELECT log_token_premium FROM d04").fetchdf()["log_token_premium"]
qs = np.linspace(0, 1, 501)
ecdf_vals = np.quantile(logtp, qs)
H02, e02 = np.histogram(logtp, bins=200)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].bar((e02[:-1] + e02[1:]) / 2, H02, width=(e02[1] - e02[0]), color="#3b6fa0")
axes[0].axvline(tp_stats["logtp_median"], color="crimson", ls="--",
                 label=f"median={tp_stats['logtp_median']:.5f}")
axes[0].set_xlabel("log(TP)", fontproperties=FP); axes[0].set_ylabel("빈도", fontproperties=FP)
axes[0].set_title("logTP 분포", fontproperties=FP); axes[0].legend(prop=FP)
axes[1].plot(ecdf_vals, qs, color="#3b6fa0")
axes[1].axvline(0, color="gray", ls=":"); axes[1].axhline(0.5, color="gray", ls=":")
axes[1].set_xlabel("log(TP)", fontproperties=FP); axes[1].set_ylabel("누적확률 (ECDF)", fontproperties=FP)
axes[1].set_title("logTP 누적분포", fontproperties=FP)
fig.suptitle("F02 — TP / logTP 분포 및 ECDF", fontproperties=FP)
FIGURE_MANIFEST["F02"] = save_fig(fig, "F02_tp_distribution_ecdf")
SUMMARY["tp_logtp"] = {k: (float(v) if hasattr(v, "item") else v) for k, v in tp_stats.to_dict().items()}
SUMMARY["rq1_annotation"] = {"bit_identical_to_nb08": bool(match), "frozen_median_logtp": RQ1_FROZEN_MEDIAN_LOGTP}


tp_median       1.333333e+00
logtp_median    2.876821e-01
logtp_mean      2.851768e-01
logtp_sd        2.220955e-01
dt_median       5.000000e+00
dt_mean         6.952042e+00
n_gt1           3.375095e+06
n_lt1           2.641750e+05
n_eq1           1.967180e+05
n               3.835988e+06
Name: 0, dtype: float64
RQ1_ANNOTATION_MATCH_BIT_IDENTICAL = True | ln(4/3) = 0.28768207245178085


### 한국어 수치 해석

`median(TP) = 1.3333333333333333` = 정확히 `4/3`, `median(logTP) = 0.28768207245178085` =
`ln(4/3)`으로, **NB08의 frozen 결과(`RQ1_PRIMARY_INFERENCE_PASS`)와 bit-identical**하다.
`TP > 1`인 pair가 3,375,095건(87.99%),
`TP < 1`이 264,175건, 정확히 `TP = 1`인 tie가
196,718건(5.13%)이다.
`token_difference`(ΔT = T_KO − T_EN) 중앙값은 5다.

**분포 해석 (F02)**: 히스토그램이 정수비 격자(4/3, 3/2, 2/1 등) 근처에서 뚜렷한 봉우리를
보이며, 이는 NB08 closeout이 이미 지적한 "bootstrap CI가 점질량에서 degenerate하는" 현상의
시각적 근거다. ECDF는 0 부근에서 급격히 상승하지 않고 완만해, 상당한 tie 질량(5.13%)이
존재함을 보여준다.

**연구적 의미**: RQ1의 결과가 이 notebook에서 독립적으로 재계산한 전집단 값과 정확히
일치함을 시각적·수치적으로 재확인했다. 이는 새로운 추론이 아니라 이미 닫힌 결과의 주석이다.

**해석 한계**: 이 절은 유의성이나 CI를 다시 계산하지 않는다. p-value, bootstrap CI, sign
test 등은 전부 NB08(`ssot_nb01/`)의 소관이며 여기서 재현하지 않는다.


## 5. 정확 분해 재검증 (Section D, RQ2, F04)

### SSOT / RQ 대응

Research Question:
RQ2 — `logTP = logCR + logBDR + logCP`의 정확 분해. **이 notebook이 canonical 결과다.**

SSOT section:
§8 (exact decomposition identity)

Canonical input:
D-04 `log_code_point_ratio`, `log_byte_density_ratio`, `log_compression_penalty`, `log_token_premium`

Physical variables:
`log_code_point_ratio`(logCR), `log_byte_density_ratio`(logBDR), `log_compression_penalty`(logCP)

Research purpose:
전체 N에 대해 항등식 오차를 재검증하고, 세 성분의 분포를 F04로 시각화한다.

Allowed claim:
항등식이 부동소수점 오차 범위 내에서 성립한다는 기술적 사실.

Prohibited claim:
세 성분에 대한 개별 추론(CAVEAT-03) — NB08의 primary inference는 logTP에만 적용되며
성분별로 자동 전이되지 않는다.


In [6]:

decomp = con.execute('''
    SELECT max(abs(log_token_premium - (log_code_point_ratio + log_byte_density_ratio + log_compression_penalty))) AS max_err,
           median(log_code_point_ratio) AS logCR_med, median(log_byte_density_ratio) AS logBDR_med,
           median(log_compression_penalty) AS logCP_med
    FROM d04
''').fetchdf().iloc[0]
print(decomp)
assert decomp["max_err"] < 1e-9, f"DECOMPOSITION_IDENTITY_VIOLATION: {decomp['max_err']}"
print(f"EXACT_DECOMPOSITION_REVALIDATED over N={n_rows:,}: max|residual| = {decomp['max_err']:.3e}")

decomp_df = con.execute("SELECT log_code_point_ratio, log_byte_density_ratio FROM d04").fetchdf()
H04, xe04, ye04 = np.histogram2d(decomp_df["log_code_point_ratio"], decomp_df["log_byte_density_ratio"], bins=100)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(xe04, ye04, np.log1p(H04.T), cmap="magma", shading="auto")
ax.axvline(0, color="white", lw=0.5, alpha=0.6); ax.axhline(0, color="white", lw=0.5, alpha=0.6)
ax.set_xlabel("log(CodePointRatio)", fontproperties=FP)
ax.set_ylabel("log(ByteDensityRatio)", fontproperties=FP)
ax.set_title("F04 — 정확 분해: logCR vs logBDR (밀도)", fontproperties=FP)
cb = fig.colorbar(im, ax=ax); cb.set_label("log(1+빈도)", fontproperties=FP)
FIGURE_MANIFEST["F04"] = save_fig(fig, "F04_exact_decomposition")
SUMMARY["decomposition"] = {k: float(v) for k, v in decomp.to_dict().items()}


max_err       8.881784e-16
logCR_med    -7.604509e-01
logBDR_med    8.960880e-01
logCP_med     1.751543e-01
Name: 0, dtype: float64
EXACT_DECOMPOSITION_REVALIDATED over N=3,835,988: max|residual| = 8.882e-16


### 한국어 수치 해석

전체 N=3,835,988에 대해 재계산한 항등식 최대 오차는
`8.882e-16`로 부동소수점 정밀도 한계 수준이다(G2
forensic adjudication의 `8.88e-16`과 동일 자릿수). 세 성분의 중앙값은
`logCR=-0.7605`, `logBDR=0.8961`,
`logCP=0.1752`이며 이는 V2 checkpoint의 역사적 기술값(-0.7605/+0.8961/+0.1752,
소수 4자리)과 반올림 일치한다 — 다만 그 일치는 **교차검증**일 뿐 V2를 authority로 인용한 것이
아니다. 이 notebook의 값은 D-04에서 새로 계산되었다.

**분포 해석 (F04)**: 밀도가 `logCR<0, logBDR>0` 사분면에 크게 쏠려 있다 — 이는 다음 절(E)의
representation reversal과 직접 연결된다.

**연구적 의미**: RQ2가 이 notebook에서 canonical하게 확정되었다.

**해석 한계 (CAVEAT-03)**: `logCR`/`logBDR`/`logCP`는 기술자(descriptor)이며, NB08의 primary
inference(median(logTP)>0)는 logTP에만 적용된다. 성분별 유의성/추론적 결론은 존재하지 않는다.


## 6. 표현 구조: representation reversal, script 구성, byte-density vs compression (Section E, F05)

### SSOT / RQ 대응

Research Question:
RQ3(surface-form 기술 구조) — **여기서는 기술만. `M1 vs M0`의 조건부 설명력은 NB09 소관.**

SSOT section:
§12.2 (D-02 script/whitespace/length group), §21 (compositional reference coding)

Canonical input:
D-02 script share 컬럼, D-04 decomposition 컬럼

Physical variables:
`ko_hangul_share`, `en_latin_share` 등 script share, `log_byte_density_ratio`, `log_compression_penalty`

Research purpose:
CR<1 & BDR>1 "representation reversal" 빈도와 byte-density/compression 관계를 기술한다.

Allowed claim:
관측된 부호 패턴의 빈도 기술.

Prohibited claim:
이 패턴이 TP를 "설명"한다는 조건부/증분 주장 (NB09 M1 vs M0 소관).


In [7]:

reversal = con.execute('''
    SELECT
      sum(CASE WHEN log_code_point_ratio<0 THEN 1 ELSE 0 END)*1.0/count(*) AS p_cr_lt1,
      sum(CASE WHEN log_byte_density_ratio>0 THEN 1 ELSE 0 END)*1.0/count(*) AS p_bdr_gt1,
      sum(CASE WHEN log_code_point_ratio<0 AND log_byte_density_ratio>0 THEN 1 ELSE 0 END) AS reversal_n,
      sum(CASE WHEN log_code_point_ratio<0 AND log_byte_density_ratio>0
               AND log_compression_penalty>0 AND log_token_premium>0 THEN 1 ELSE 0 END) AS sign_pattern_n,
      sum(CASE WHEN abs(log_compression_penalty) > abs(log_code_point_ratio+log_byte_density_ratio)
               THEN 1 ELSE 0 END) AS cp_dominant_n,
      count(*) AS n
    FROM d04
''').fetchdf().iloc[0]
print(reversal)
print(f"REVERSAL_SHARE = {reversal['reversal_n']/reversal['n']*100:.2f}%")
print(f"SIGN_PATTERN_SHARE (CR-,BDR+,CP+,TP+) = {reversal['sign_pattern_n']/reversal['n']*100:.2f}%")
print(f"CP_DOMINANT_SHARE (|logCP|>|logCR+logBDR|) = {reversal['cp_dominant_n']/reversal['n']*100:.2f}%")

script_med = con.execute('''
    SELECT median(ko_hangul_share) AS ko_hangul, median(ko_whitespace_density) AS ko_ws,
           median(en_latin_share) AS en_latin, median(en_whitespace_density) AS en_ws
    FROM d02
''').fetchdf().iloc[0]
print(script_med)

bdr_cp_df = con.execute("SELECT log_byte_density_ratio, log_compression_penalty FROM d04").fetchdf()
H05, xe05, ye05 = np.histogram2d(bdr_cp_df["log_byte_density_ratio"], bdr_cp_df["log_compression_penalty"], bins=100)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(xe05, ye05, np.log1p(H05.T), cmap="cividis", shading="auto")
ax.set_xlabel("log(ByteDensityRatio)", fontproperties=FP)
ax.set_ylabel("log(CompressionPenalty)", fontproperties=FP)
ax.set_title("F05 — Byte-Density vs Compression Penalty (밀도)", fontproperties=FP)
cb = fig.colorbar(im, ax=ax); cb.set_label("log(1+빈도)", fontproperties=FP)
FIGURE_MANIFEST["F05"] = save_fig(fig, "F05_byte_density_vs_compression")
SUMMARY["representation_reversal"] = {k: float(v) for k, v in reversal.to_dict().items()}


p_cr_lt1          9.966913e-01
p_bdr_gt1         9.996530e-01
reversal_n        3.823048e+06
sign_pattern_n    2.863051e+06
cp_dominant_n     2.073073e+06
n                 3.835988e+06
Name: 0, dtype: float64
REVERSAL_SHARE = 99.66%
SIGN_PATTERN_SHARE (CR-,BDR+,CP+,TP+) = 74.64%
CP_DOMINANT_SHARE (|logCP|>|logCR+logBDR|) = 54.04%


ko_hangul    0.724638
ko_ws        0.214286
en_latin     0.805556
en_ws        0.160000
Name: 0, dtype: float64


### 한국어 수치 해석

`log_code_point_ratio<0`(CR<1) 비율은 99.67%,
`log_byte_density_ratio>0`(BDR>1) 비율은 99.97%이며, 두 조건을 **동시에** 만족하는
representation reversal 비율은 **99.66%**다
(N=3,823,048/3,835,988).
공통 부호 패턴 `(CR-, BDR+, CP+, TP+)`는 74.64%,
`|logCP| > |logCR+logBDR|`는 54.04%다.

**⚠ 불일치 보고 (억지로 맞추지 않음).** 이 값(reversal ≈ 99.7%)은
비canonical 역사적 V2 checkpoint가 기록한 "representation reversal ≈ 71.05%"와 **크게 다르다**.
이 notebook은 D-04 전집단에서 `log_code_point_ratio`/`log_byte_density_ratio`의 부호를 직접
재계산했으며 (마진값 각각 99.67%/99.97%), V2가 어떤 정의·표본·컬럼을 사용했는지 이 notebook에서
독립적으로 재현할 수 없었다. 두 수치를 억지로 일치시키지 않고 **불일치 자체를 기록**한다 —
G5 독립 감사가 채택한 원칙과 동일하다. Director/Claude-B 검토가 필요한 항목으로
`[EDA-REF-DISC-01]`에 등록한다(아래 anomaly register 참조).

**분포 해석 (F05)**: byte-density와 compression penalty가 함께 양의 영역에 밀집해, 두 메커니즘이
독립적이지 않고 동반됨을 보여준다.

**연구적 의미**: F04/F05가 M1(surface 블록)의 입력이 될 원시 분포를 제공한다.

**해석 한계**: 이 절은 어떤 조건부/증분 설명력도 주장하지 않는다(NB09 M1 vs M0 소관).


## 7. 형태론 분포 (Section F, RQ4 기술)

### SSOT / RQ 대응

Research Question:
RQ4(형태론 증분 설명력) — **여기서는 분포 기술만. M2 vs M1 증분 설명력은 NB09 소관.**

SSOT section:
§12.3 (D-03 physical schema), §13.6/§13.7 (density/ratio 정의)

Canonical input:
D-03 `morpheme_density`, `particle_ratio`, `ending_ratio`, `deriv_affix_ratio`, `function_morpheme_ratio`

Physical variables:
위 5개 컬럼 (M2/M2A 블록의 physical column)

Research purpose:
형태론 지표의 전집단 분포를 기술한다.

Allowed claim:
분포 기술(중앙값·분위수).

Prohibited claim:
형태론이 TP를 유발한다는 인과 주장 — 절대 금지.


In [8]:

morph = con.execute('''
    SELECT median(morpheme_density) AS density_med, median(particle_ratio) AS particle_med,
           median(ending_ratio) AS ending_med, median(deriv_affix_ratio) AS deriv_med,
           median(function_morpheme_ratio) AS func_med,
           min(morpheme_density) AS density_min, max(morpheme_density) AS density_max
    FROM d03
''').fetchdf().iloc[0]
print(morph)
SUMMARY["morphology"] = {k: float(v) for k, v in morph.to_dict().items()}


density_med      2.214286
particle_med     0.157895
ending_med       0.175439
deriv_med        0.065217
func_med         0.342105
density_min      0.500000
density_max     30.000000
Name: 0, dtype: float64


### 한국어 수치 해석

`morpheme_density` 중앙값 2.2143, `particle_ratio` 중앙값 0.1579,
`ending_ratio` 중앙값 0.1754, `deriv_affix_ratio` 중앙값 0.0652,
`function_morpheme_ratio` 중앙값 0.3421이다. `morpheme_density`의 범위는
`[0.5, 30.0]`로, 최댓값 30.0은 §12절(extreme audit)에서 별도
감사한다.

**분포 해석**: 형태론 지표들이 넓은 범위에 걸쳐 있으며 다수가 0을 포함한다(particle_ratio,
ending_ratio, deriv_affix_ratio는 조사/어미/파생접사가 없는 문장에서 0).

**연구적 의미**: M2/M2A의 입력 분포를 제공한다. `function_morpheme_ratio`는 M2A에서만,
`particle_ratio`+`ending_ratio`는 M2에서만 사용되며 동일 모형에 함께 들어가지 않는다
(G5 협약).

**해석 한계**: 형태론 지표와 TP의 관계는 이 절에서 전혀 다루지 않는다. RQ4의 증분 설명력은
NB09 M2 vs M1의 소관이다.


## 8. D-05 정규식 청킹 메커니즘 기술자 (Section G, RQ5 기술)

### SSOT / RQ 대응

Research Question:
RQ5(정규식 청킹 메커니즘 기여) — **여기서는 분포 기술만. M3 vs M2 조건부 기여는 NB09 소관.**

SSOT section:
§12.5 (D-05 physical schema)

Canonical input:
D-05 `ko/en_chunk_count`, `ko/en_tokens_per_chunk`, `ko/en_chunk_type_share_*`

Physical variables:
`ko_chunk_count`, `en_chunk_count`, `ko_tokens_per_chunk`, `en_tokens_per_chunk`,
chunk-type share 컬럼

Research purpose:
NB06/D-05가 이미 보고한 chunk-count/tokens-per-chunk 비대칭을 이 notebook에서
독립적으로 재계산해 F06(→NB09)의 사전 기술 자료로 남긴다.

Allowed claim:
`ln(N_ko/N_en) = ln(C_ko/C_en) + ln(density_ratio)` 항등식과 관측 패턴의 기술.

Prohibited claim:
정규식 청킹이 TP를 유발한다는 주장 — D-05 README도 명시적으로 금지.


In [9]:

d05_stats = con.execute('''
    SELECT avg(ko_chunk_count) AS ko_cc_mean, avg(en_chunk_count) AS en_cc_mean,
           avg(ko_tokens_per_chunk) AS ko_tpc_mean, avg(en_tokens_per_chunk) AS en_tpc_mean
    FROM d05
''').fetchdf().iloc[0]
decomp_ln = con.execute('''
    SELECT avg(ln(t.ko_token_count*1.0/t.en_token_count)) AS ln_N_ratio,
           avg(ln(c.ko_chunk_count*1.0/c.en_chunk_count)) AS ln_C_ratio,
           avg(ln((t.ko_token_count*1.0/c.ko_chunk_count)/(t.en_token_count*1.0/c.en_chunk_count))) AS ln_density_ratio,
           max(abs(ln(t.ko_token_count*1.0/t.en_token_count)
                   - (ln(c.ko_chunk_count*1.0/c.en_chunk_count)
                      + ln((t.ko_token_count*1.0/c.ko_chunk_count)/(t.en_token_count*1.0/c.en_chunk_count)))
               )) AS max_resid
    FROM d04 t JOIN d05 c USING(pair_id)
''').fetchdf().iloc[0]
print(d05_stats); print(decomp_ln)
assert decomp_ln["max_resid"] < 1e-9
SUMMARY["d05_mechanism"] = {k: float(v) for k, v in d05_stats.to_dict().items()}
SUMMARY["d05_ln_decomposition"] = {k: float(v) for k, v in decomp_ln.to_dict().items()}


ko_cc_mean     13.561120
en_cc_mean     19.498922
ko_tpc_mean     2.018622
en_tpc_mean     1.036495
Name: 0, dtype: float64
ln_N_ratio          2.851768e-01
ln_C_ratio         -3.671581e-01
ln_density_ratio    6.523349e-01
max_resid           4.718448e-16
Name: 0, dtype: float64


### 한국어 수치 해석

평균 `ko_chunk_count`=13.56, `en_chunk_count`=19.50(KO가 더 적음);
평균 `ko_tokens_per_chunk`=2.02, `en_tokens_per_chunk`=1.04(KO가
훨씬 큼). 항등식 `ln(N_ko/N_en) = ln(C_ko/C_en) + ln(density_ratio)`을 재검증한 결과 잔차 최대
4.72e-16로 정확히 닫힌다: `+0.2852 = -0.3672 + 0.6523`.
이는 NB06/D-05 result package의 수치(13.56/19.50, 2.02/1.04, -0.3672/+0.6523)와 독립적으로
일치한다(교차검증, authority 아님).

**분포 해석**: 두 항의 부호가 반대다 — KO는 chunk 수가 적지만(음의 항) chunk당 확장이 훨씬
크다(양의 항). 두 효과가 서로 상쇄하면서도 순 효과는 양(+)이다.

**연구적 의미**: 이 관측은 아래 EDA-REF-M3-01(§10)에서 다루는 G5 M3-01 review trigger의
배경이 된다: chunk 수가 pair 길이와 강하게 연관되어 M3에서 collinearity가 발생했다.

**해석 한계**: 이 절은 정규식 청킹이 TP를 "유발한다"는 어떤 주장도 하지 않는다. 언어학적
형태론(D-03) ≠ tokenizer 정규식 청킹(D-05) ≠ 최종 subword 토큰화(D-04)라는 SSOT의 3중 구분을
유지한다.


## 9. Source / Domain / Direction / Length 이질성 (Section H, RQ6 기술, F03)

### SSOT / RQ 대응

Research Question:
RQ6(이질성) — **기술 통계만. 모형 기반 효과는 이후 단계 소관.**

SSOT section:
§20.1/§20.2 (source-domain 해석 제약)

Canonical input:
D-01 `source_id`, `domain`, `translation_direction`, `length_stratum` × D-04 `log_token_premium`

Physical variables:
위 4개 categorical/strat 변수 × `log_token_premium`

Research purpose:
층별 기술 통계(median logTP)를 F03으로 시각화한다.

Allowed claim:
"관측된 층별 median(logTP)은 다르다."

Prohibited claim:
순수 source 효과, 순수 domain 효과 (G5 ID-03) — source_domain_cell은 관측 층 통제일 뿐이다.


In [10]:

by_cell = con.execute('''
    SELECT substr(source_id,1,3) AS src, domain, count(*) AS n, median(t.log_token_premium) AS logtp_med
    FROM d01 d JOIN d04 t USING(pair_id)
    GROUP BY substr(source_id,1,3), domain ORDER BY 1,2
''').fetchdf()
by_direction = con.execute('''
    SELECT translation_direction, count(*) AS n, median(t.log_token_premium) AS logtp_med
    FROM d01 d JOIN d04 t USING(pair_id) GROUP BY translation_direction ORDER BY n DESC
''').fetchdf()
by_length = con.execute('''
    SELECT length_stratum, count(*) AS n, median(t.log_token_premium) AS logtp_med
    FROM d01 d JOIN d04 t USING(pair_id) GROUP BY length_stratum ORDER BY length_stratum
''').fetchdf()
print(by_cell.to_string(index=False)); print(by_direction.to_string(index=False)); print(by_length.to_string(index=False))

pooled_median = con.execute("SELECT median(log_token_premium) FROM d04").fetchone()[0]
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = [f"{row.src}-{row.domain}" for row in by_cell.itertuples()]
bars = ax.bar(labels, by_cell["logtp_med"], color="#4c8caa")
for b, n in zip(bars, by_cell["n"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height()+0.005, f"n={n:,}", ha="center", fontsize=8, fontproperties=FP)
ax.axhline(pooled_median, color="crimson", ls="--", label="pooled median")
ax.set_ylabel("median(logTP)", fontproperties=FP)
ax.set_title("F03 — source×domain별 기술 통계 TP (관측 지지 구조, 순수 효과 아님)", fontproperties=FP)
ax.legend(prop=FP)
plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontproperties=FP)
FIGURE_MANIFEST["F03"] = save_fig(fig, "F03_domain_descriptive_tp")
SUMMARY["heterogeneity"] = {"by_cell": by_cell.to_dict("records"), "by_direction": by_direction.to_dict("records"),
                             "by_length": by_length.to_dict("records"), "pooled_median": float(pooled_median)}


src     domain       n  logtp_med
025   dialogue  516162   0.251314
025    general  804291   0.251314
025      other 1165510   0.305382
026      other  990120   0.293348
026 technology  359905   0.346276
translation_direction       n  logtp_med
             KO_TO_EN 2512152   0.300105
             EN_TO_KO 1273289   0.262364
              UNKNOWN   50547   0.325422
length_stratum      n  logtp_med
            Q1 840667   0.287682
            Q2 858560   0.287682
            Q3 802735   0.287682
            Q4 725825   0.325422
            Q5 608201   0.273293


### 한국어 수치 해석

source×domain 층별 median(logTP): 025-dialogue: 0.2513(n=516,162); 025-general: 0.2513(n=804,291); 025-other: 0.3054(n=1,165,510); 026-other: 0.2933(n=990,120); 026-technology: 0.3463(n=359,905). pooled median(0.28768)은
어느 한 층과도 정확히 일치하지 않는다. 방향별: KO_TO_EN: 0.3001; EN_TO_KO: 0.2624; UNKNOWN: 0.3254. 길이 층별(length_stratum): Q2: 0.2877(n=858,560); Q1: 0.2877(n=840,667); Q3: 0.2877(n=802,735); Q4: 0.3254(n=725,825); Q5: 0.2733(n=608,201).

**분포 해석 (F03)**: 층 간 median(logTP)이 025-dialogue/general(≈0.251)에서
026-technology(≈0.346)까지 폭넓게 분포한다 — pooled 값은 이 층들의 단순 평균이 아니라
표본 크기 가중 혼합이다.

**연구적 의미**: 이질성이 크다는 사실 자체는 RQ6의 기술적 관찰이며, NB09가 M0(pair_log_size +
source_domain_cell + translation_direction)에서 다룰 대상이다.

**해석 한계 (G5 ID-03/ID-04 재확인)**: `source_domain_cell` 계수는 순수 source 효과도 순수
domain 효과도 아니다 — 이 cohort에서 source와 domain은 분리 식별되지 않는다(5개 관측 셀,
`other`만 두 source에 걸침). `translation_direction`의 대비는 025 층 내부 변이에서만
식별된다(026은 EN_TO_KO 관측이 0건). 이 두 제약은 §11에서 다시 시각화한다.


## 10. 극단값 / 경계 감사 (Section I, F07)

### SSOT / RQ 대응

Research Question:
없음(품질/경계 기술) — anomaly register의 근거.

SSOT section:
R3 caveat (`ssot/2026-08-17_1730_..._G2_G3_G4_FINAL_ADJUDICATION.md` §13, §19)

Canonical input:
D-03 `eojeol_count`, `morpheme_density`; D-04 `token_premium`

Physical variables:
`eojeol_count`, `morpheme_density`, `token_premium`

Research purpose:
극단값을 **삭제하지 않고** 전체 분포 + zoom/log/robust panel로 기술하며, `eojeol_count=1` 밴드를
반드시 재검증한다.

Allowed claim:
관측 빈도·비율의 기술.

Prohibited claim:
`eojeol_count=1` 밴드를 "언어적 복잡도"로 해석하는 것 — G2-G4가 이미 금지했다
(analyzer artifact of short input, not linguistic signal).


In [11]:

eojeol1 = con.execute('''
    SELECT sum(CASE WHEN eojeol_count=1 THEN 1 ELSE 0 END) AS n_eojeol1, count(*) AS n,
           median(CASE WHEN eojeol_count=1 THEN morpheme_density END) AS density_med_eojeol1
    FROM d03
''').fetchdf().iloc[0]
eojeol1_share = eojeol1["n_eojeol1"] / eojeol1["n"]
print(f"EOJEOL1_COUNT={int(eojeol1['n_eojeol1']):,}  EOJEOL1_SHARE={eojeol1_share*100:.4f}%")

tp_extremes = con.execute('''
    SELECT quantile_cont(token_premium, [0.001, 0.01, 0.99, 0.999]) AS tp_pcts,
           sum(CASE WHEN token_premium >= 10 THEN 1 ELSE 0 END) AS n_tp_ge10,
           sum(CASE WHEN token_premium <= 0.1 THEN 1 ELSE 0 END) AS n_tp_le01
    FROM d04
''').fetchdf().iloc[0]
print(tp_extremes)

logtp_full = con.execute("SELECT log_token_premium FROM d04").fetchdf()["log_token_premium"]
H, e = np.histogram(logtp_full, bins=200)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar((e[:-1]+e[1:])/2, H, width=(e[1]-e[0]), color="#8a8f9a")
p999 = tp_extremes["tp_pcts"]
ax.axvline(math.log(p999[0]), color="orange", ls=":", label="TP p0.1/p99.9 경계")
ax.axvline(math.log(p999[-1]), color="orange", ls=":")
ax.set_xlabel("log(TP)", fontproperties=FP); ax.set_ylabel("빈도", fontproperties=FP)
ax.set_title("F07 — 정제된 극단값 감사 (원문 미포함, 집계 기술량만)", fontproperties=FP)
ax.legend(prop=FP)
FIGURE_MANIFEST["F07"] = save_fig(fig, "F07_sanitized_extreme_audit")

ANOMALIES.append({
    "id": "EDA-REF-EOJEOL1-01", "observed": "eojeol_count = 1",
    "n": int(eojeol1["n_eojeol1"]), "share": float(eojeol1_share),
    "rule": "eojeol_count=1 -> morpheme_density 분모(eojeol_count)=1, 밀도/particle_ratio 최댓값 밴드",
    "status": "EXPECTED_BOUNDARY", "primary_cohort": "INCLUDED",
    "ssot_relevance": "R3 caveat, G2-G4 최종 adjudication §13/§19",
    "interpretation": "공백 없는 단문/숫자열/코드열에서 형태소 분석기가 하나의 어절로 처리한 결과",
    "not_established": "언어적 복잡도, 형태론적 특이성",
    "downstream": "NB07/NB08 분포·극단값 패널에서 analyzer artifact로 취급, NB09 모형 입력에서 배제하지 않음(정책상 포함)"
})
ANOMALIES.append({
    "id": "EDA-REF-TPEXT-01", "observed": "token_premium >= 10 또는 <= 0.1",
    "n": int(tp_extremes["n_tp_ge10"] + tp_extremes["n_tp_le01"]),
    "share": float((tp_extremes["n_tp_ge10"] + tp_extremes["n_tp_le01"]) / n_rows),
    "rule": "TP p0.1/p99.9 밖 극단 pair",
    "status": "REVIEW_REQUIRED", "primary_cohort": "INCLUDED",
    "ssot_relevance": "§17.3 대표본 해석 원칙 — 극단값이 median을 흔들지 않음(NB08 tie-aware sign test로 이미 확인)",
    "interpretation": "매우 짧은 pair 또는 번역 품질 이상치일 가능성 — 원문 없이 집계 수준에서만 기술",
    "not_established": "이 pair들이 오류라는 결론",
    "downstream": "로컬 전용 감사 대상(원문 미공개), NB11 sensitivity 후보"
})
SUMMARY["eojeol1"] = {"n": int(eojeol1["n_eojeol1"]), "share": float(eojeol1_share)}
SUMMARY["tp_extremes"] = {"n_tp_ge10": int(tp_extremes["n_tp_ge10"]), "n_tp_le01": int(tp_extremes["n_tp_le01"])}


EOJEOL1_COUNT=42,096  EOJEOL1_SHARE=1.0974%


tp_pcts      [0.5555555555555556, 0.75, 2.25, 2.90909090909...
n_tp_ge10                                                 13.0
n_tp_le01                                                  1.0
Name: 0, dtype: object


### 한국어 수치 해석

`[EDA-REF-EOJEOL1-01]`
Observed: `eojeol_count = 1`
N/share: 42,096 / 1.0974%
Rule: `morpheme_density = morpheme_count / eojeol_count`이므로 분모가 1이면 밀도가 morpheme_count와
같아져 최댓값 밴드에 집중된다(density_max=30.0인 행들이 이 밴드에 속함).
Status: `EXPECTED_BOUNDARY`
Primary cohort: `INCLUDED`
SSOT relevance: G2-G4 adjudication R3 — "공백 없는 numeral/alphanumeric 문자열 또는 띄어쓰기 없는
1-eojeol 절"로 이미 감사됨.
Interpretation: 형태소 분석기가 공백 부재 문자열을 1개 어절로 처리한 구조적 결과.
Not established: **언어적 복잡도가 아니다** — 명시적으로 그렇게 부르지 않는다.
Downstream: NB07/NB08 분포·극단 패널의 표준 caveat, NB09 모형에서 배제하지 않음(정책상 포함).

`[EDA-REF-TPEXT-01]` — TP p0.1/p99.9 밖 극단(F07 참조): 13건 TP≥10,
1건 TP≤0.1. Status: `REVIEW_REQUIRED`(로컬 감사 필요, 원문 비공개).

**분포 해석 (F07)**: logTP 히스토그램에 p0.1/p99.9 경계를 표시했다 — 극단값이 자동 삭제되지
않고 primary cohort에 포함된 채로 시각화된다.

**연구적 의미**: SSOT의 "never auto-delete" 원칙을 물리적으로 확인시킨다.

**해석 한계**: 이 절의 감사는 sanitized 집계 수준이며, 개별 pair의 원문은 어디에도 커밋되지
않는다.


## 11. G5 Review Register 시각화 — EDA-REF-M3-01 (chunk 규모 vs 절대 길이)

### SSOT / RQ 대응

Research Question:
없음 — G5가 발견한 `M3-01` review trigger(reparameterization review)를 시각적으로 설명하는
descriptive 절. **추론적 결론이 아니다.**

SSOT section:
G5 collinearity report (`outputs/reports/G5_COLLINEARITY_v001.json`) §7.1 — M3 조건수 134.57,
`pair_log_size` VIF 1,252.61

Canonical input:
D-02 `ko/en_codepoint_count`(→`pair_log_size`), D-05 `ko/en_chunk_count`(→`*_chunk_count_log`)

Physical variables:
`pair_log_size` = 0.5·(ln(ko_codepoint_count)+ln(en_codepoint_count)) (derived, G5 §3),
`ko_chunk_count_log` = ln(ko_chunk_count), `en_chunk_count_log` = ln(en_chunk_count) (derived, G5 §3.1)

Research purpose:
`M3-01`이 왜 fired했는지(chunk 수가 절대 길이와 거의 비례)를 시각적으로 보인다.

Allowed claim:
"chunk 수는 tokenizer 정규식 수준의 기술자이며, pair 절대 길이와 강한 연관을 보인다 —
이것이 G5 M3-01 reparameterization review의 근거다."

Prohibited claim:
인과적 축소성(causal redundancy) 주장, NB09 변수 제거 허가 — **이 notebook은 어떤 변수도
드롭할 권한이 없다.**


In [12]:

m3_df = con.execute('''
    SELECT 0.5*(ln(d.ko_codepoint_count)+ln(d.en_codepoint_count)) AS pair_log_size,
           ln(c.ko_chunk_count) AS ko_chunk_count_log, ln(c.en_chunk_count) AS en_chunk_count_log
    FROM d02 d JOIN d05 c USING(pair_id)
''').fetchdf()
r_size_ko = np.corrcoef(m3_df["pair_log_size"], m3_df["ko_chunk_count_log"])[0,1]
r_size_en = np.corrcoef(m3_df["pair_log_size"], m3_df["en_chunk_count_log"])[0,1]
r_ko_en = np.corrcoef(m3_df["ko_chunk_count_log"], m3_df["en_chunk_count_log"])[0,1]
print(f"pearson(pair_log_size, ko_chunk_count_log) = {r_size_ko:.4f}")
print(f"pearson(pair_log_size, en_chunk_count_log) = {r_size_en:.4f}")
print(f"pearson(ko_chunk_count_log, en_chunk_count_log) = {r_ko_en:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
panels = [
    ("pair_log_size", "ko_chunk_count_log", r_size_ko),
    ("pair_log_size", "en_chunk_count_log", r_size_en),
    ("ko_chunk_count_log", "en_chunk_count_log", r_ko_en),
]
for ax, (xcol, ycol, corr) in zip(axes, panels):
    H, xe, ye = np.histogram2d(m3_df[xcol], m3_df[ycol], bins=80)
    im = ax.pcolormesh(xe, ye, np.log1p(H.T), cmap="viridis", shading="auto")
    ax.set_xlabel(xcol); ax.set_ylabel(ycol); ax.set_title(f"r={corr:.4f}")
fig.suptitle("EDA-REF-M3-01 — chunk 규모 vs 절대 길이 (G5 M3-01 reparameterization review 근거)", fontproperties=FP)
FIGURE_MANIFEST["EDA-REF-M3-01"] = save_fig(fig, "EDA_REF_M3_01_chunk_scale_vs_length")
SUMMARY["eda_ref_m3_01"] = {"pearson_size_ko": float(r_size_ko), "pearson_size_en": float(r_size_en),
                              "pearson_ko_en": float(r_ko_en)}


pearson(pair_log_size, ko_chunk_count_log) = 0.9585
pearson(pair_log_size, en_chunk_count_log) = 0.9686
pearson(ko_chunk_count_log, en_chunk_count_log) = 0.9356


### 한국어 수치 해석

`pair_log_size`와 `ko_chunk_count_log`의 Pearson r = 0.9585,
`en_chunk_count_log`와의 r = 0.9686, 두 언어의 `chunk_count_log` 간
r = 0.9356이다. 세 값 모두 매우 높다(0.93 이상).

**분포 해석**: 세 산점(밀도) 모두 강한 선형 추세를 보인다 — chunk 수는 tokenizer 정규식
수준의 기술자이며, pair의 절대 텍스트 길이(`pair_log_size`)와 거의 비례한다.

**연구적 의미**: 이것이 바로 G5 collinearity report가 M3에서 조건수 134.57, `pair_log_size`
VIF 1,252.61을 보고한 **이유**다 — chunk-scale 항이 M0의 길이 항과 M1의 길이비 항을 거의
재현하기 때문이다. G5는 이를 rank 결손이 아닌 **reparameterization review**(`M3-01`)로
분류했다(rank는 45/45로 full).

**해석 한계**: 이 관측은 **인과적 축소성을 확립하지 않는다.** `M3-01`은 review trigger이지
자동 삭제가 아니며, 이 notebook은 NB09에서 어떤 변수를 드롭할지 결정할 권한이 없다 —
그 재매개변수화(reparameterization) 결정은 NB09가 M3을 적합하기 **전에** 별도로 내려야 한다.


## 12. G5 Review Register 시각화 — EDA-REF-SM-01 (script mixing 중복성)

### SSOT / RQ 대응

Research Question:
없음 — G5 `SM-01` review trigger(representative-feature review)의 시각적 근거. NB09
대표 feature 선택은 **여기서 하지 않는다.**

SSOT section:
G5 collinearity report §7.2 — `script_mixing` family, `|Spearman ρ| ≥ 0.95`

Canonical input:
D-02 `ko/en_script_type_count`, `ko/en_script_switch_count`

Physical variables:
`ko_script_type_count`, `ko_script_switch_count`, `en_script_type_count`, `en_script_switch_count`

Research purpose:
KO/EN 각각의 Spearman ρ를 이 notebook에서 새로 계산하고 결합분포를 시각화한다.

Allowed claim:
두 구성 개념(존재하는 script 종류 수 vs 전환 횟수)의 관측된 통계적 근접성 기술.

Prohibited claim:
NB09에서 어느 feature를 대표로 쓸지 이 절에서 결정하는 것 — 명시적으로 금지.


In [13]:

sm_df = con.execute('''
    SELECT ko_script_type_count, ko_script_switch_count, en_script_type_count, en_script_switch_count
    FROM d02
''').fetchdf()
rho_ko, _ = stats.spearmanr(sm_df["ko_script_type_count"], sm_df["ko_script_switch_count"])
rho_en, _ = stats.spearmanr(sm_df["en_script_type_count"], sm_df["en_script_switch_count"])
print(f"Spearman rho KO (script_type_count ~ script_switch_count) = {rho_ko:.10f}")
print(f"Spearman rho EN (script_type_count ~ script_switch_count) = {rho_en:.10f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
H4, xe4, ye4 = np.histogram2d(sm_df["ko_script_type_count"], sm_df["ko_script_switch_count"],
                                bins=[np.arange(0.5, 8.5, 1), np.arange(-0.5, 40.5, 1)])
H5, xe5, ye5 = np.histogram2d(sm_df["en_script_type_count"], sm_df["en_script_switch_count"],
                                bins=[np.arange(0.5, 8.5, 1), np.arange(-0.5, 40.5, 1)])
im1 = axes[0].pcolormesh(xe4, ye4, np.log1p(H4.T), cmap="magma", shading="auto")
axes[0].set_xlabel("ko_script_type_count", fontproperties=FP); axes[0].set_ylabel("ko_script_switch_count", fontproperties=FP)
axes[0].set_title(f"KO: Spearman rho={rho_ko:.4f}", fontproperties=FP)
im2 = axes[1].pcolormesh(xe5, ye5, np.log1p(H5.T), cmap="magma", shading="auto")
axes[1].set_xlabel("en_script_type_count", fontproperties=FP); axes[1].set_ylabel("en_script_switch_count", fontproperties=FP)
axes[1].set_title(f"EN: Spearman rho={rho_en:.4f}", fontproperties=FP)
fig.suptitle("EDA-REF-SM-01 — script_type_count vs script_switch_count (G5 SM-01 representative-feature review)", fontproperties=FP)
FIGURE_MANIFEST["EDA-REF-SM-01"] = save_fig(fig, "EDA_REF_SM_01_script_mixing_redundancy")
SUMMARY["eda_ref_sm_01"] = {"ko_rho": float(rho_ko), "en_rho": float(rho_en)}
print("PRE_NB09_REPRESENTATIVE_FEATURE_REVIEW_REQUIRED")


Spearman rho KO (script_type_count ~ script_switch_count) = 0.9910090163
Spearman rho EN (script_type_count ~ script_switch_count) = 0.9993520546


PRE_NB09_REPRESENTATIVE_FEATURE_REVIEW_REQUIRED


### 한국어 수치 해석

이 notebook에서 독립적으로 재계산한 Spearman ρ: KO = 0.9910090163, EN = 0.9993520546.
G5 adjudication(0.9910090163 / 0.9993520546)·독립 감사(0.9910090163 / 0.9993520546, mid-rank
방법)와 **완전히 일치**한다(교차검증, 이 notebook 자체가 authority임).

**분포 해석**: 두 언어 모두에서 `script_type_count`(존재하는 script 종류 수, 상한 있음)와
`script_switch_count`(script 간 전환 횟수, 상한 실질적으로 큼)가 거의 단조적으로 함께
움직인다 — EN이 특히 더 강하다(0.9994).

**연구적 의미**: 두 구성 개념 — "몇 종류의 script가 존재하는가"와 "몇 번 전환이 일어나는가" —
이 이 cohort에서 사실상 순위-동치(rank-equivalent)에 가깝다. G5는 이를 `SM-01`
representative-feature review로 분류했다(VIF는 완만함, ≈9 — 순수 선형 중복은 아님).

**해석 한계**: `PRE_NB09_REPRESENTATIVE_FEATURE_REVIEW_REQUIRED` — 이 notebook은 어느 feature를
NB09의 대표로 사용할지 **결정하지 않는다.** 그 결정은 구성 개념(construct) 근거로, NB09
착수 전에 별도로 내려야 한다(계수 크기로 사후 결정하지 않는다).


## 13. Identifiability 지지 구조 시각화 — G5 ID-03 / ID-04

### SSOT / RQ 대응

Research Question:
없음 — G5가 이미 형식적으로 판정한 identifiability NOTE(`ID-03`, `ID-04`)를 시각적으로
노출한다.

SSOT section:
§20.2, §32 T-03/T-04

Canonical input:
D-01 `source_id`, `domain`, `translation_direction` × D-04 `pair_id` spine

Physical variables:
`source_id`, `domain`, `translation_direction`

Research purpose:
빈 셀·근-단일 셀·UNKNOWN 카운트를 heatmap으로 노출해 지지 구조의 한계를 시각적으로
드러낸다.

Allowed claim:
"관측된 지지 구조는 다음과 같다" — 표/heatmap 기술.

Prohibited claim:
순수 source 효과, 순수 domain 효과, `cell × direction` 상호작용 추정 (G5: 이 상호작용은
estimable하지 않으며 어떤 모형에도 도입되지 않는다).


In [14]:

sd_cells = con.execute('''
    SELECT substr(source_id,1,3) AS src, domain, count(*) AS n
    FROM d01 JOIN d04 USING(pair_id) GROUP BY substr(source_id,1,3), domain
''').fetchdf()
srcs = sorted(sd_cells["src"].unique())
doms = ["dialogue", "general", "other", "technology"]
mat = np.zeros((len(srcs), len(doms)))
for row in sd_cells.itertuples():
    mat[srcs.index(row.src), doms.index(row.domain)] = row.n

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(mat, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(doms))); ax.set_xticklabels(doms, fontproperties=FP)
ax.set_yticks(range(len(srcs))); ax.set_yticklabels(srcs, fontproperties=FP)
for i in range(len(srcs)):
    for j in range(len(doms)):
        v = mat[i, j]
        ax.text(j, i, f"{int(v):,}" if v > 0 else "—", ha="center", va="center",
                 color="white" if v > mat.max()/2 else "black", fontsize=9, fontproperties=FP)
ax.set_title("G5 ID-03 — source x domain 지지 구조 (빈 셀 = 미관측 조합)", fontproperties=FP)
FIGURE_MANIFEST["EDA-REF-ID-03"] = save_fig(fig, "EDA_REF_ID_03_source_domain_support")

cell_dir = con.execute('''
    SELECT substr(source_id,1,3) AS src, domain, translation_direction, count(*) AS n
    FROM d01 JOIN d04 USING(pair_id)
    GROUP BY substr(source_id,1,3), domain, translation_direction
''').fetchdf()
cells_list = sorted(set(zip(cell_dir["src"], cell_dir["domain"])))
dirs = ["KO_TO_EN", "EN_TO_KO", "UNKNOWN"]
mat2 = np.zeros((len(cells_list), len(dirs)))
for row in cell_dir.itertuples():
    mat2[cells_list.index((row.src, row.domain)), dirs.index(row.translation_direction)] = row.n

fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(mat2, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(dirs))); ax.set_xticklabels(dirs, fontproperties=FP)
ax.set_yticks(range(len(cells_list))); ax.set_yticklabels([f"{s}-{d}" for s, d in cells_list], fontproperties=FP)
for i in range(len(cells_list)):
    for j in range(len(dirs)):
        v = mat2[i, j]
        ax.text(j, i, f"{int(v):,}" if v > 0 else "0", ha="center", va="center",
                 color="white" if v > mat2.max()/2 else "black", fontsize=8, fontproperties=FP)
ax.set_title("G5 ID-04 — source_domain_cell x translation_direction (빈칸/근-단일 노출)", fontproperties=FP)
FIGURE_MANIFEST["EDA-REF-ID-04"] = save_fig(fig, "EDA_REF_ID_04_cell_direction_support")

n_empty = int((mat2 == 0).sum())
near_singleton = [(cells_list[i], dirs[j], int(mat2[i, j])) for i in range(len(cells_list)) for j in range(len(dirs))
                   if 0 < mat2[i, j] <= 50]
print("empty combinations:", n_empty)
print("near-singleton (<=50):", near_singleton)
SUMMARY["identifiability"] = {
    "source_domain_cells": int(len(srcs) * len(doms)) and sd_cells.to_dict("records"),
    "empty_cell_direction_combinations": n_empty,
    "near_singleton_combinations": near_singleton,
}


empty combinations: 3
near-singleton (<=50): [(('025', 'general'), 'UNKNOWN', 31), (('026', 'other'), 'UNKNOWN', 1)]


### 한국어 수치 해석

`ID-03`: 관측된 source×domain 조합은 5개뿐이다 — `other` domain만 두 source(025/026)에
걸쳐 있고, `dialogue`/`general`은 025 전용, `technology`는 026 전용이다. 이는 additive
`source + domain` 설계가 관측된 5개 셀에서 정확히 saturate됨(파라미터 5개, rank 5, 잔여
자유도 0)을 의미하며, source와 domain의 독립 주효과는 분리 식별되지 않는다.

`ID-04`: `source_domain_cell × translation_direction` 15개 조합 중 3개가 완전히
비어있고(빈 칸), 2개가 근-단일(025-general×UNKNOWN=31, 026-other×UNKNOWN=1)이다. **026
계열은 `EN_TO_KO` 관측이 전혀 없다.**

**분포 해석 (heatmap)**: 두 heatmap 모두 이론적 full factorial과 실제 관측 지지 구조 사이의
간극을 시각적으로 드러낸다.

**연구적 의미**: `source_domain_cell`의 계수는 관측된 층 간 조건부 대비(conditional
contrast)로만 해석 가능하다. `translation_direction`의 대비는 사실상 025 층 내부 변이에서만
식별된다.

**해석 한계**: `cell × direction` 상호작용은 이 설계에서 **estimable하지 않으며 어떤 모형에도
도입되지 않는다**(G5 확정 사항). 이 notebook은 그 판정을 재검정하지 않고 시각적으로만
노출한다.


## 14. Canonical Figure Manifest 및 Deferred Figures

### SSOT / RQ 대응

Research Question:
없음 — figure dependency 명세.

SSOT section:
§16.2 (canonical figure F01-F09 contract) — 이 fork는 PDF §16.2를 직접 재확인하지 못했으므로,
아래 dependency 매핑은 이번 세션에서 읽은 G5/RQ1/G2-G4 문서가 인용한 section 번호를
따른 것이며 `(§# 미독립검증)`으로 표시한다.

Canonical input:
이 notebook이 생성한 F01/F02/F03/F04/F05/F07

Physical variables:
해당 없음

Research purpose:
어떤 figure가 이 단계에서 dependency-ready이고 어떤 것이 이후 단계로 미뤄지는지 명시한다.

Allowed claim:
F01/F02/F03/F04/F05/F07는 이 notebook에서 canonical하게 생성되었다.

Prohibited claim:
NB08-RQ1-Vxx, NB06_D05_Vxx를 Fxx로 재명명하는 것 — 절대 하지 않았다.


In [15]:

FIGURE_DEPENDENCY = {
    "F01": {"desc": "KO/EN 토큰 수 관계", "dependency": "D-04", "status": "GENERATED"},
    "F02": {"desc": "TP/logTP 분포 및 ECDF", "dependency": "D-04", "status": "GENERATED"},
    "F03": {"desc": "domain 기술 TP", "dependency": "D-04 + D-01 metadata", "status": "GENERATED"},
    "F04": {"desc": "정확 분해", "dependency": "D-02/D-04", "status": "GENERATED"},
    "F05": {"desc": "byte-density vs compression", "dependency": "D-02/D-04", "status": "GENERATED"},
    "F06": {"desc": "형태론 partial effect", "dependency": "NB09 M2 vs M1", "status": "DEFERRED_BY_DEPENDENCY"},
    "F07": {"desc": "정제된 극단값 감사", "dependency": "local audit + privacy policy", "status": "GENERATED"},
    "F08": {"desc": "source/domain 설명 forest", "dependency": "NB09", "status": "DEFERRED_BY_DEPENDENCY"},
    "F09": {"desc": "Track B", "dependency": "NB12", "status": "DEFERRED_BY_DEPENDENCY"},
}
for fid, meta in FIGURE_DEPENDENCY.items():
    print(fid, meta["status"], "-", meta["desc"], "<-", meta["dependency"])

REFERENCE_ONLY_FIGURES = {
    "NB08-RQ1-Vxx": "results/rq1-publication-visuals-20260817 패키지 — result-communication용, canonical F01-F09 아님",
    "NB06_D05_Vxx": "results/nb06-d05-reproducibility-20260817 패키지 — result-communication용, canonical F01-F09 아님",
}
print(REFERENCE_ONLY_FIGURES)
SUMMARY["figure_dependency"] = FIGURE_DEPENDENCY
SUMMARY["reference_only_figures"] = REFERENCE_ONLY_FIGURES


F01 GENERATED - KO/EN 토큰 수 관계 <- D-04
F02 GENERATED - TP/logTP 분포 및 ECDF <- D-04
F03 GENERATED - domain 기술 TP <- D-04 + D-01 metadata
F04 GENERATED - 정확 분해 <- D-02/D-04
F05 GENERATED - byte-density vs compression <- D-02/D-04
F06 DEFERRED_BY_DEPENDENCY - 형태론 partial effect <- NB09 M2 vs M1
F07 GENERATED - 정제된 극단값 감사 <- local audit + privacy policy
F08 DEFERRED_BY_DEPENDENCY - source/domain 설명 forest <- NB09
F09 DEFERRED_BY_DEPENDENCY - Track B <- NB12
{'NB08-RQ1-Vxx': 'results/rq1-publication-visuals-20260817 패키지 — result-communication용, canonical F01-F09 아님', 'NB06_D05_Vxx': 'results/nb06-d05-reproducibility-20260817 패키지 — result-communication용, canonical F01-F09 아님'}


### 한국어 수치 해석

F01/F02/F03/F04/F05/F07 6개가 이 notebook에서 생성되어 `GENERATED` 상태다. F06(형태론
partial effect)·F08(설명 forest)·F09(Track B)는 각각 NB09 M2 vs M1, NB09 explanatory
model, NB12 Track B에 의존하므로 `DEFERRED_BY_DEPENDENCY`로 명시적으로 표기하고 placeholder
이미지를 생성하지 않았다.

**분포 해석**: 해당 없음(dependency 명세 절).

**연구적 의미**: NB09/NB12가 착수될 때 무엇을 만들어야 하는지 명확한 체크리스트를 제공한다.

**해석 한계**: `NB08-RQ1-Vxx`와 `NB06_D05_Vxx`는 참고용 결과-소통 figure이며 canonical
F01-F09가 아니다 — 어디에서도 재명명하지 않았다.


## 15. Anomaly Register 및 Validation 확정

### SSOT / RQ 대응

Research Question:
없음 — 산출물 무결성 확정.

SSOT section:
§17 (validation 요건, 이전 세션 표기 계승)

Canonical input:
이 notebook 전체 실행 상태

Physical variables:
해당 없음

Research purpose:
anomaly register를 확정하고, artifact/manifest 일관성을 검증하고, Korean font smoke test를
재확인한다.

Allowed claim:
검증 통과/실패 상태의 기술.

Prohibited claim:
없음 — 이 절은 순수 검증이다.


In [16]:

# EDA-REF-DISC-01: representation reversal 불일치 (V2 71.05% vs 이 notebook 재계산치)
reversal_recomputed = con.execute('''
    SELECT sum(CASE WHEN log_code_point_ratio<0 AND log_byte_density_ratio>0 THEN 1 ELSE 0 END) AS n,
           count(*) AS total
    FROM d04
''').fetchdf().iloc[0]
ANOMALIES.append({
    "id": "EDA-REF-DISC-01",
    "observed": "representation reversal share (CR<1 & BDR>1)",
    "n": int(reversal_recomputed["n"]), "share": float(reversal_recomputed["n"]/reversal_recomputed["total"]),
    "rule": "log_code_point_ratio<0 AND log_byte_density_ratio>0, D-04 전집단 재계산",
    "status": "REVIEW_REQUIRED",
    "primary_cohort": "INCLUDED",
    "ssot_relevance": "비canonical 역사적 V2 checkpoint(docs/results/pre_g5_v2/README.md)가 기록한 71.05%와 불일치",
    "interpretation": "이 notebook은 D-04에서 독립적으로 새로 계산했으며 marginal P(CR<1)=99.67%, P(BDR>1)=99.97%로 매우 높음",
    "not_established": "두 수치 중 어느 쪽이 옳은지에 대한 판정 -- 이 notebook은 V2의 원 방법론을 재현할 수 없었다",
    "downstream": "Director/Claude-B가 V2 산출 방법(정의·컬럼·표본)을 대조 검토할 것을 권고"
})

# 헤드리스 검증: Korean font smoke
smoke_fig, smoke_ax = plt.subplots(figsize=(4, 2))
smoke_ax.set_title("한글 폰트 확인 - 마이너스 부호", fontproperties=FP)
smoke_ax.plot([-2, -1, 0, 1, 2], [1, 2, 0, 2, 1])
smoke_meta = save_fig(smoke_fig, "NB07_KOREAN_FONT_SMOKE")
print("KOREAN_FONT_SMOKE_OK, family=", FONT_INFO["family"])

# manifest <-> figure consistency
import os
fig_files_on_disk = sorted(p.name for p in FIG_DIR.glob("*.png"))
fig_files_in_manifest = sorted(f"{v['name']}.png" for v in FIGURE_MANIFEST.values())
print("figures on disk:", len(fig_files_on_disk), "| in manifest (this run):", len(fig_files_in_manifest))

VALIDATION = {
    "headless_status": "PASS",
    "korean_plot_font": FONT_INFO["family"],
    "decomposition_max_residual_lt_1e9": True,
    "artifact_identity_5_of_5": True,
    "cohort_n": n_rows,
    "pair_set_md5": pair_set_md5,
    "rq1_bit_identical": SUMMARY.get("rq1_annotation", {}).get("bit_identical_to_nb08"),
    "anomaly_total": len(ANOMALIES),
    "review_required": sum(1 for a in ANOMALIES if a["status"] == "REVIEW_REQUIRED"),
    "possible_defect": sum(1 for a in ANOMALIES if a["status"] == "POSSIBLE_DEFECT"),
    "raw_text_committed": False,
    "legacy_casebook_numerical_source": False,
}
print(json.dumps(VALIDATION, indent=1, ensure_ascii=False))
SUMMARY["validation"] = VALIDATION


KOREAN_FONT_SMOKE_OK, family= NanumGothic
figures on disk: 11 | in manifest (this run): 10
{
 "headless_status": "PASS",
 "korean_plot_font": "NanumGothic",
 "decomposition_max_residual_lt_1e9": true,
 "artifact_identity_5_of_5": true,
 "cohort_n": 3835988,
 "pair_set_md5": "d9660d654ee449e4d0c23a0070225274",
 "rq1_bit_identical": true,
 "anomaly_total": 3,
 "review_required": 2,
 "possible_defect": 0,
 "raw_text_committed": false,
 "legacy_casebook_numerical_source": false
}


### 한국어 수치 해석

`[EDA-REF-DISC-01]` — representation reversal 재계산치가 비canonical V2 checkpoint의
71.05%와 불일치함을 기록했다(위 §6에서 이미 상세 기술). 억지로 재현·수정하지 않고 그대로
register에 남긴다.

Korean font smoke test는 `NanumGothic`(`/usr/share/fonts/truetype/nanum/NanumGothic.ttf`)으로
통과했다 — 한글 title/음수 부호가 포함된 최소 figure를 렌더링해 tofu box 없이 저장을
확인했다.

**분포 해석**: 해당 없음.

**연구적 의미**: 이 notebook의 산출물이 headless 재실행 가능하고, 결측/원문 유출이 없으며,
레거시 casebook의 수치에 의존하지 않았음을 자체 검증했다.

**해석 한계**: `EDA-REF-DISC-01`은 미해결 상태로 다음 단계(Director/Claude-B) 검토가
필요하다 — 이 notebook 스스로 해결하지 않는다.


## 16. 산출물 기록 (Reports / Manifests)

D-02~D-05에서 계산된 모든 수치를 `outputs/reports/NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001.json`,
anomaly register를 `outputs/reports/NB07_ANOMALY_REGISTER_v001.json`, figure 목록을
`outputs/manifests/NB07_FIGURE_MANIFEST_v001.json`에 기록한다. 원문(raw KO/EN text),
pair_id 목록, token/chunk 문자열은 어디에도 쓰지 않는다.


In [17]:

import datetime as dt
from zoneinfo import ZoneInfo

RUN_TS = dt.datetime.now(tz=ZoneInfo("Asia/Seoul")).isoformat(timespec="seconds")

descriptive_summary = {
    "artifact_id": "NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001",
    "generated_at_kst": RUN_TS,
    "execution_base_sha": "9d99e13026b89dcf7d8846d0c105a811f64274bc",
    "notebook": "notebooks/07_eda_and_decomposition.ipynb",
    "cohort_n": n_rows,
    "pair_set_md5": pair_set_md5,
    "artifact_identity": identity_report,
    "summary": SUMMARY,
}
(REPORT_DIR / "NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001.json").write_text(
    json.dumps(descriptive_summary, ensure_ascii=False, indent=1, default=str) + "\n", encoding="utf-8")

anomaly_register = {
    "artifact_id": "NB07_ANOMALY_REGISTER_v001",
    "generated_at_kst": RUN_TS,
    "policy": "NEVER_AUTO_DELETE -- 모든 extreme은 primary cohort에 INCLUDED 상태로 유지",
    "anomalies": ANOMALIES,
    "totals": {
        "anomaly_total": len(ANOMALIES),
        "review_required": sum(1 for a in ANOMALIES if a["status"] == "REVIEW_REQUIRED"),
        "possible_defect": sum(1 for a in ANOMALIES if a["status"] == "POSSIBLE_DEFECT"),
        "expected_boundary": sum(1 for a in ANOMALIES if a["status"] == "EXPECTED_BOUNDARY"),
        "plausible_extreme": sum(1 for a in ANOMALIES if a["status"] == "PLAUSIBLE_EXTREME"),
    },
}
(REPORT_DIR / "NB07_ANOMALY_REGISTER_v001.json").write_text(
    json.dumps(anomaly_register, ensure_ascii=False, indent=1, default=str) + "\n", encoding="utf-8")

figure_manifest = {
    "artifact_id": "NB07_FIGURE_MANIFEST_v001",
    "generated_at_kst": RUN_TS,
    "canonical_figures": FIGURE_MANIFEST,
    "figure_dependency": SUMMARY.get("figure_dependency", {}),
    "reference_only_figures": SUMMARY.get("reference_only_figures", {}),
    "korean_font": FONT_INFO,
    "korean_font_smoke": smoke_meta,
}
(MANIFEST_DIR / "NB07_FIGURE_MANIFEST_v001.json").write_text(
    json.dumps(figure_manifest, ensure_ascii=False, indent=1, default=str) + "\n", encoding="utf-8")

print("WROTE:")
print(" -", REPORT_DIR / "NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001.json")
print(" -", REPORT_DIR / "NB07_ANOMALY_REGISTER_v001.json")
print(" -", MANIFEST_DIR / "NB07_FIGURE_MANIFEST_v001.json")


WROTE:
 - /home/sieg/projects-wsl/KOEN_nb07_20260818/outputs/reports/NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001.json
 - /home/sieg/projects-wsl/KOEN_nb07_20260818/outputs/reports/NB07_ANOMALY_REGISTER_v001.json
 - /home/sieg/projects-wsl/KOEN_nb07_20260818/outputs/manifests/NB07_FIGURE_MANIFEST_v001.json


## 17. Claim Boundary — 종결

**허용되는 진술.**

- D-02~D-05 canonical artifact의 SHA-256이 5/5 검증되었고, N=3,835,988, pair-set
  `d9660d654ee449e4d0c23a0070225274`이 전 절에서 동일하게 유지되었다.
- `median(logTP) = ln(4/3) = 0.28768207245178085`가 이 notebook의 독립 재계산과 NB08 frozen
  결과에서 bit-identical하다(RQ1 annotation).
- `logTP = logCR + logBDR + logCP` 항등식이 전집단에서 부동소수점 정밀도 한계 내로
  재검증되었다(RQ2, canonical).
- RQ3(surface)/RQ4(형태론)/RQ5(D-05 mechanism)/RQ6(이질성)의 **기술 통계**가 canonical하게
  확정되었다.
- G5 review register(`M3-01`, `SM-01`)와 identifiability NOTE(`ID-03`, `ID-04`)를 시각적으로
  노출했다.
- `eojeol_count=1`(1.10%) 밴드를 재검증했고, `EDA-REF-DISC-01`(representation reversal
  불일치)을 정직하게 보고했다.

**금지되는 진술.**

- 어떤 causal 언어도 사용하지 않았다.
- RQ1을 재검정하지 않았다(NB08이 이미 닫음).
- RQ3/RQ4/RQ5/RQ6의 조건부/증분/메커니즘 설명력을 주장하지 않았다(NB09 소관).
- M3-01/SM-01 review trigger를 근거로 어떤 변수도 드롭하거나 대표 feature를 선정하지 않았다.
- source/domain의 순수 효과, `cell × direction` 상호작용을 추정하지 않았다.
- D-02~D-05를 재생성하거나 수정하지 않았다 — 읽기 전용이었다.

```
NB07_CANONICAL_DESCRIPTIVE_EDA_COMPLETE
READY_FOR_CLAUDE_B_NB07_AUDIT
DO_NOT_MERGE_MAIN
```
